# Export QAT Model to Hex Format in Google Colab

This notebook exports the QAT (Quantization-Aware Training) model from checkpoints to Verilog hex format using the export.py script.

## 1. Mount Google Drive

Mount Google Drive to access your project files and save outputs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone Repository and Install Dependencies

In [ ]:
import os
import sys

# Option 1: Clone from GitHub
# os.chdir('/content')
# !git clone https://github.com/orpheus016/6g-pa-gan-dpd.git
# project_dir = '/content/6g-pa-gan-dpd'

# Option 2: Use from Google Drive (if already uploaded)
# Adjust path to where you have the project in Drive
project_dir = '/content/drive/MyDrive/6g-pa-gan-dpd'

# Change to project directory
os.chdir(project_dir)
sys.path.insert(0, project_dir)

print(f"Working directory: {os.getcwd()}")
print(f"Files in project: {os.listdir('.')[:10]}")

In [ ]:
# Install required dependencies
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu -q
!pip install pyyaml numpy -q

print("✓ Dependencies installed")

## 3. Import Required Libraries

In [ ]:
import torch
import numpy as np
import yaml
from pathlib import Path
import struct

# Import export functions
from export import load_checkpoint, export_verilog_hex, export_binary, export_c_header
from utils.quantization import quantize_weights_fixed_point

print("✓ Libraries imported")

## 4. Configure Export Parameters

Set the checkpoint file, output directory, and export format.

In [ ]:
# Configuration
checkpoint_path = 'checkpoints/fla_qat_best_26jan.pth'  # Change this to your checkpoint
config_path = 'config/config.yaml'
output_dir = 'rtl/weights_qat_export'  # Output directory
export_formats = ['hex']  # Export only hex format

# Create output directory if it doesn't exist
Path(output_dir).mkdir(parents=True, exist_ok=True)

print(f"Checkpoint: {checkpoint_path}")
print(f"Output directory: {output_dir}")
print(f"Export formats: {export_formats}")

## 5. Load Configuration and Checkpoint

In [ ]:
# Load configuration
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print(f"Config loaded from {config_path}")
print(f"Model config: {config['model']['generator']}")

# Load checkpoint and model
print(f"\nLoading checkpoint: {checkpoint_path}")
model = load_checkpoint(checkpoint_path, config)
print(f"✓ Model loaded successfully")
print(f"Model: {model.__class__.__name__}")

# Get quantization bits
bits = config['quantization']['weight_bits']
print(f"Quantization bits: {bits}")

## 6. Extract and Quantize Weights

In [ ]:
print("\n" + "="*60)
print("EXTRACTING AND QUANTIZING WEIGHTS")
print("="*60)

# Extract and quantize weights
weights = {}
all_weights = []

for name, param in model.named_parameters():
    if 'weight' in name or 'bias' in name:
        w_float = param.detach().cpu().numpy()
        w_fixed = quantize_weights_fixed_point(
            torch.tensor(w_float),
            num_bits=bits
        ).numpy()
        weights[name] = w_fixed
        all_weights.extend(w_fixed.flatten())
        print(f"  {name}: shape {list(param.shape)} -> {len(w_fixed.flatten())} elements")

all_weights = np.array(all_weights)

print(f"\n✓ Total parameters: {len(all_weights)}")
print(f"✓ Quantization format: Q1.{bits-1} fixed-point")

## 7. Export to Hex Format (Verilog)

In [ ]:
if 'hex' in export_formats:
    print("\nExporting to hex format...")
    hex_path = Path(output_dir) / 'weights_qat.hex'
    export_verilog_hex(all_weights, hex_path, bits)
    print(f"✓ Hex file saved to: {hex_path}")

## 8. Verify Export Results

In [ ]:
import os

print("\n" + "="*60)
print("EXPORT SUMMARY")
print("="*60)

# Summary
total_params = sum(p.numel() for p in model.parameters())
memory_bytes = total_params * bits // 8
hex_file_path = Path(output_dir) / 'weights_qat.hex'

print(f"\n✅ Export Complete!")
print(f"\nModel Statistics:")
print(f"  Total parameters: {total_params}")
print(f"  Weight bits: {bits}")
print(f"  Memory per bank: {memory_bytes} bytes")

print(f"\nOutput Files:")
print(f"  📄 {hex_file_path}")

# Check file size
if hex_file_path.exists():
    file_size = os.path.getsize(hex_file_path)
    print(f"  File size: {file_size} bytes")

print(f"\nOutput directory: {Path(output_dir).absolute()}")
print("\n✓ Ready for FPGA deployment!")